# Self-Training (ST) for Few-Shot Disaster Tweet Classification

This notebook runs **Self-Training (ST)** experiments using a BERT-based model (BERTweet) for classifying disaster-related tweets into 10 humanitarian categories.

## Algorithm Overview

Self-Training is a semi-supervised learning approach that leverages a small set of labeled data together with a large pool of unlabeled data:

1. **Supervised fine-tuning (base model):** A pre-trained BERTweet model is fine-tuned on the small labeled dataset. To reduce sensitivity to random initialization, `N_base` independent runs are performed and the best model (by validation macro-F1) is selected.

2. **Iterative pseudo-labeling:** For each self-training iteration:
   - A subset of unlabeled examples (`sample_size`) is drawn from the unlabeled pool.
   - The model assigns pseudo-labels to these examples. With the `"uniform"` sampling scheme, pseudo-labeled instances are selected uniformly at random (no uncertainty estimation).
   - A new training set is formed by combining the original labeled data with the selected pseudo-labeled data (`unsup_size` instances).
   - The model is re-trained on this combined set, with a weighted loss: 50% supervised + 50% unsupervised. Confidence-weighted loss reweighting is controlled by `alpha`.
   - Early stopping on validation macro-F1 prevents overfitting.

3. **Evaluation:** The best checkpoint is evaluated on a held-out test set, reporting macro-F1 and Expected Calibration Error (ECE with `n_bins=10`).

## Humanitarian Categories

The full label space contains up to 10 classes, but **not every disaster dataset has all 10**. Some events may only have 7, 8, or 9 classes depending on the types of tweets observed. The notebook automatically detects the actual number of classes per dataset.

| ID | Category |
|---|---|
| 0 | Caution and advice |
| 1 | Displaced people and evacuations |
| 2 | Infrastructure and utility damage |
| 3 | Injured or dead people |
| 4 | Missing or found people |
| 5 | Not humanitarian |
| 6 | Other relevant information |
| 7 | Requests or urgent needs |
| 8 | Rescue, volunteering, or donation effort |
| 9 | Sympathy and support |

## Datasets

Each disaster folder under `data/` contains:
- `labeled_{k}_set{s}.tsv` — few-shot labeled splits (k = 5, 10, 25, 50 per class; s = 1, 2, 3)
- `unlabeled_{k}_set{s}.tsv` — corresponding unlabeled pools
- `{disaster}_dev.tsv` — validation split
- `{disaster}_test.tsv` — test split

## Key Hyperparameters

| Parameter | Default | Description |
|---|---|---|
| `sample_scheme` | `"uniform"` | Sampling strategy for pseudo-label selection (uniform = standard ST) |
| `sup_epochs` | 18 | Max epochs for supervised fine-tuning (with early stopping, patience=3) |
| `unsup_epochs` | 12 | Number of self-training iterations |
| `N_base` | 3 | Number of random initializations for base model selection |
| `T` | 7 | Number of MC Dropout forward passes (not used for uniform scheme) |
| `alpha` | 0.1 | Confidence loss reweighting factor |
| `sample_size` | 1800 | Unlabeled instances sampled for uncertainty evaluation per iteration |
| `unsup_size` | 1000 | Pseudo-labeled instances used per self-training iteration |
| `sup_batch_size` | 16 | Batch size for supervised training |
| `unsup_batch_size` | 64 | Batch size for self-training on pseudo-labeled data |

## Setup

Import dependencies and configure the environment. The `PYTHONHASHSEED` environment variable must be set for reproducibility — it is used as the global seed throughout the pipeline.

In [2]:
import os
import sys
import json
import logging
import time
import numpy as np
import pandas as pd
import random
import contextlib
import io

# Set seeds BEFORE importing torch/transformers
os.environ["PYTHONHASHSEED"] = "42"
os.environ["TQDM_DISABLE"] = "1"  # Suppress tqdm progress bars from ust.py
GLOBAL_SEED = int(os.environ["PYTHONHASHSEED"])
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

import torch
torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)

# Add the project root to the path so we can import project modules
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DATA_ROOT = os.path.join(PROJECT_ROOT, "data")    

# Suppress noisy logging from httpx, transformers, and UST internals
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("UST").setLevel(logging.WARNING)
import transformers
transformers.logging.set_verbosity_error()

from transformers import AutoConfig, AutoTokenizer
from custom_dataset import CustomDataset_tracked, CustomDataset
from ust import train_model

# Logging — only show warnings and above
logger = logging.getLogger("UST")
logging.basicConfig(level=logging.WARNING)

print(f"Global seed: {GLOBAL_SEED}")
print(f"Project root: {PROJECT_ROOT}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Global seed: 42
Project root: D:\Workspace\UST
CUDA available: True
GPU: NVIDIA RTX A6000


## Helper Functions

Define the label mapping and dataset loading utilities. These mirror what `run_ust.py` does but are defined here so the notebook is self-contained.

**Important:** Not every disaster has all 10 classes. `detect_classes()` scans the train, dev, and test splits to build the actual label-to-id mapping for each disaster, so the model only predicts the classes that are actually present.

In [3]:
# Full 10-class humanitarian label mapping (superset)
FULL_LABEL_TO_ID = {
    "caution_and_advice": 0,
    "displaced_people_and_evacuations": 1,
    "infrastructure_and_utility_damage": 2,
    "injured_or_dead_people": 3,
    "missing_or_found_people": 4,
    "not_humanitarian": 5,
    "other_relevant_information": 6,
    "requests_or_urgent_needs": 7,
    "rescue_volunteering_or_donation_effort": 8,
    "sympathy_and_support": 9,
}


def detect_classes(disaster, train_file, data_root="data"):
    """Detect the actual classes present in a disaster dataset.
    
    Scans train, dev, and test TSV files to collect all unique class labels,
    then builds a contiguous label-to-id mapping (0, 1, 2, ...).
    
    Returns:
        label_to_id (dict): mapping from class name to integer id
        n_classes (int): number of unique classes
    """
    base = os.path.join(data_root, disaster)
    files = [
        os.path.join(base, f"labeled_{train_file}.tsv"),
        os.path.join(base, f"{disaster}_dev.tsv"),
        os.path.join(base, f"{disaster}_test.tsv"),
    ]
    all_labels = set()
    for f in files:
        if os.path.exists(f):
            df = pd.read_csv(f, sep="\t")
            all_labels.update(df["class_label"].dropna().unique())

    # Build a contiguous mapping, preserving the canonical order from FULL_LABEL_TO_ID
    label_to_id = {}
    idx = 0
    for label in FULL_LABEL_TO_ID:
        if label in all_labels:
            label_to_id[label] = idx
            idx += 1

    return label_to_id, len(label_to_id)


def get_dataset(path, tokenizer, label_to_id, labeled=True):
    """Load a TSV file into a CustomDataset_tracked instance."""
    df = pd.read_csv(path, sep="\t")
    text_list = []
    labels_list = []
    ids_list = []
    for _, row in df.iterrows():
        if pd.isna(row["tweet_text"]):
            continue
        text_list.append(row["tweet_text"])
        labels_list.append(label_to_id[row["class_label"]])
        ids_list.append(row["tweet_id"])
    return CustomDataset_tracked(text_list, labels_list, ids_list, tokenizer, labeled=labeled)


def load_disaster_datasets(disaster, train_file, tokenizer, label_to_id, data_root="data"):
    """Load train, dev, test, and unlabeled datasets for a given disaster."""
    base = os.path.join(data_root, disaster)
    ds_train = get_dataset(os.path.join(base, f"labeled_{train_file}.tsv"), tokenizer, label_to_id)
    ds_dev = get_dataset(os.path.join(base, f"{disaster}_dev.tsv"), tokenizer, label_to_id)
    ds_test = get_dataset(os.path.join(base, f"{disaster}_test.tsv"), tokenizer, label_to_id)
    ds_unlabeled = get_dataset(os.path.join(base, f"unlabeled_{train_file}.tsv"), tokenizer, label_to_id, labeled=False)
    print(f"  Train: {len(ds_train)} | Dev: {len(ds_dev)} | Test: {len(ds_test)} | Unlabeled: {len(ds_unlabeled)}")
    return ds_train, ds_dev, ds_test, ds_unlabeled


print(f"Full label space: {len(FULL_LABEL_TO_ID)} classes")
print(f"Labels: {list(FULL_LABEL_TO_ID.keys())}")

Full label space: 10 classes
Labels: ['caution_and_advice', 'displaced_people_and_evacuations', 'infrastructure_and_utility_damage', 'injured_or_dead_people', 'missing_or_found_people', 'not_humanitarian', 'other_relevant_information', 'requests_or_urgent_needs', 'rescue_volunteering_or_donation_effort', 'sympathy_and_support']


## Configuration

Set the model checkpoint, dropout rates, and other hyperparameters. For **standard Self-Training**, we use `sample_scheme = "uniform"`, which randomly selects pseudo-labeled instances without uncertainty-based filtering.

All other hyperparameters use their default values as defined in the project.

In [4]:
# Model checkpoint
PT_TEACHER_CHECKPOINT = "vinai/bertweet-base"

# Dropout configuration
HIDDEN_DROPOUT_PROB = 0.3
ATTENTION_PROBS_DROPOUT_PROB = 0.3
DENSE_DROPOUT = 0.5

# Self-Training hyperparameters
SAMPLE_SCHEME = "uniform"        # Standard ST — uniform pseudo-label selection
SUP_EPOCHS = 18                  # Max supervised fine-tuning epochs (early stopping patience=3)
UNSUP_EPOCHS = 12                # Number of self-training iterations
N_BASE = 3                       # Random initializations for base model selection
T = 7                            # MC Dropout passes (unused for uniform scheme)
ALPHA = 0.1                      # Confidence loss reweighting factor
SAMPLE_SIZE = 1800               # Unlabeled instances sampled per ST iteration
UNSUP_SIZE = 1000                # Pseudo-labeled instances used per ST iteration
SUP_BATCH_SIZE = 16
UNSUP_BATCH_SIZE = 64

# Training data splits — all 12 combinations: 4 sizes × 3 sets
LABEL_SIZES = [5, 10, 25, 50]
SETS = [1, 2, 3]
TRAIN_FILES = [f"{size}_set{s}" for size in LABEL_SIZES for s in SETS]

# Results output directory (separate from data/)
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")

# Results file naming: st_uniform_{train_file}.txt per combination
def results_file_for(train_file):
    return f"st_uniform_{train_file}"

# Build model config with custom dropout
cfg = AutoConfig.from_pretrained(PT_TEACHER_CHECKPOINT)
cfg.hidden_dropout_prob = HIDDEN_DROPOUT_PROB
cfg.attention_probs_dropout_prob = ATTENTION_PROBS_DROPOUT_PROB

# Initialize tokenizer (shared across all experiments)
tokenizer = AutoTokenizer.from_pretrained(PT_TEACHER_CHECKPOINT)

print("Configuration loaded.")
print(f"  Model: {PT_TEACHER_CHECKPOINT}")
print(f"  Sample scheme: {SAMPLE_SCHEME}")
print(f"  Train splits: {len(TRAIN_FILES)} ({', '.join(TRAIN_FILES)})")
print(f"  Results will be saved to: {RESULTS_DIR}/{{disaster}}/st_uniform_{{split}}.txt")
print(f"  Supervised epochs: {SUP_EPOCHS}, ST iterations: {UNSUP_EPOCHS}, N_base: {N_BASE}")

Configuration loaded.
  Model: vinai/bertweet-base
  Sample scheme: uniform
  Train splits: 12 (5_set1, 5_set2, 5_set3, 10_set1, 10_set2, 10_set3, 25_set1, 25_set2, 25_set3, 50_set1, 50_set2, 50_set3)
  Results will be saved to: D:\Workspace\UST\results/{disaster}/st_uniform_{split}.txt
  Supervised epochs: 18, ST iterations: 12, N_base: 3


---

## Part 1: Quick Sanity Check — Single Disaster

Before running on all datasets, we verify the pipeline works end-to-end on a **single disaster** (`california_wildfires_2018`) with the smallest labeled split (`5_set1` = 5 examples per class = 50 total).

This cell should complete relatively quickly and confirms that:
- Data loading works correctly
- The base model trains and selects the best initialization
- Self-training iterations run without errors
- Test evaluation produces valid F1 and ECE scores

If this cell runs successfully, you can proceed to the full experiment below.

In [5]:
# --- Sanity check: single disaster, single split ---
SANITY_DISASTER = "california_wildfires_2018"
SANITY_TRAIN_FILE = TRAIN_FILES[0]  # "5_set1"
DATA_ROOT = os.path.join(PROJECT_ROOT, "data")

# Detect the actual classes for this disaster
label_to_id, n_classes = detect_classes(SANITY_DISASTER, SANITY_TRAIN_FILE, data_root=DATA_ROOT)
print(f"Loading data for: {SANITY_DISASTER} (split: {SANITY_TRAIN_FILE})")
print(f"  Detected {n_classes} classes: {list(label_to_id.keys())}")

ds_train, ds_dev, ds_test, ds_unlabeled = load_disaster_datasets(
    SANITY_DISASTER, SANITY_TRAIN_FILE, tokenizer, label_to_id, data_root=DATA_ROOT
)

print(f"\nRunning Self-Training on {SANITY_DISASTER} ...")
start_time = time.time()

with contextlib.redirect_stdout(io.StringIO()):
    train_model(
        ds_train, ds_dev, ds_test, ds_unlabeled,
        PT_TEACHER_CHECKPOINT, cfg,
        model_dir=SANITY_DISASTER,
        sup_batch_size=SUP_BATCH_SIZE,
        unsup_batch_size=UNSUP_BATCH_SIZE,
        unsup_size=UNSUP_SIZE,
        sample_size=SAMPLE_SIZE,
        sample_scheme=SAMPLE_SCHEME,
        T=T,
        alpha=ALPHA,
        sup_epochs=SUP_EPOCHS,
        unsup_epochs=UNSUP_EPOCHS,
        N_base=N_BASE,
        dense_dropout=DENSE_DROPOUT,
        attention_probs_dropout_prob=ATTENTION_PROBS_DROPOUT_PROB,
        hidden_dropout_prob=HIDDEN_DROPOUT_PROB,
        results_file=results_file_for(SANITY_TRAIN_FILE),
        results_dir=RESULTS_DIR,
        data_dir=DATA_ROOT,
        temp_scaling=False,
        ls=0.0,
        n_classes=n_classes,
    )

elapsed = time.time() - start_time
print(f"Done in {elapsed/60:.1f} min")

results_path = os.path.join(RESULTS_DIR, SANITY_DISASTER, f"{results_file_for(SANITY_TRAIN_FILE)}.txt")
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    print(f"\nResults for {SANITY_DISASTER} ({SANITY_TRAIN_FILE}):")
    print(json.dumps(results, indent=2))
else:
    print(f"\nWarning: results file not found at {results_path}")

print("\nSanity check complete.")

Loading data for: california_wildfires_2018 (split: 5_set1)
  Detected 10 classes: ['caution_and_advice', 'displaced_people_and_evacuations', 'infrastructure_and_utility_damage', 'injured_or_dead_people', 'missing_or_found_people', 'not_humanitarian', 'other_relevant_information', 'requests_or_urgent_needs', 'rescue_volunteering_or_donation_effort', 'sympathy_and_support']
  Train: 50 | Dev: 752 | Test: 1461 | Unlabeled: 5113

Running Self-Training on california_wildfires_2018 ...
Done in 10.3 min

Results for california_wildfires_2018 (5_set1):
{
  "Temperature Scaling": false,
  "Label Smoothing": 0.0,
  "Best ST model": {
    "F1 before temp scaling": "0.38310464104623404",
    "ECE before temp scaling": "tensor(0.4798, device='cuda:0')",
    "T before temp scaling": "1.0"
  }
}

Sanity check complete.


---

## Part 2: Full Experiment — All Disasters × All Splits

Run Self-Training across **all 10 disaster datasets** with **all 12 labeled splits** (4 sizes × 3 sets = 120 experiments total).

Results are saved per combination to `results/{disaster}/st_uniform_{size}_set{s}.txt`. The loop automatically skips experiments that already have results, so it's safe to re-run after a crash.

In [7]:
# --- Pre-scan: check all 120 experiments (10 disasters × 12 splits) ---
ALL_DISASTERS = sorted([
    d for d in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, d))
])

pending_experiments = []  # (disaster, train_file, total_entries)
done_experiments = []     # (disaster, train_file)
skip_experiments = []

for disaster in ALL_DISASTERS:
    base = os.path.join(DATA_ROOT, disaster)
    for train_file in TRAIN_FILES:
        results_path = os.path.join(RESULTS_DIR, disaster, f"{results_file_for(train_file)}.txt")
        train_path = os.path.join(base, f"labeled_{train_file}.tsv")
        unlabeled_path = os.path.join(base, f"unlabeled_{train_file}.tsv")
        dev_path = os.path.join(base, f"{disaster}_dev.tsv")
        test_path = os.path.join(base, f"{disaster}_test.tsv")

        # Count entries
        total_entries = 0
        for path in [train_path, unlabeled_path, dev_path, test_path]:
            if os.path.exists(path):
                total_entries += len(pd.read_csv(path, sep="\t"))

        if os.path.exists(results_path):
            done_experiments.append((disaster, train_file))
        elif not os.path.exists(train_path) or not os.path.exists(unlabeled_path):
            skip_experiments.append((disaster, train_file))
        else:
            pending_experiments.append((disaster, train_file, total_entries))

total_pending_entries = sum(e for _, _, e in pending_experiments)
total_experiments = len(ALL_DISASTERS) * len(TRAIN_FILES)

# Compact summary by disaster
print(f"{'Disaster':<35} {'Done':>5} {'Pending':>8} {'Skip':>5}")
print("─" * 58)
for disaster in ALL_DISASTERS:
    n_done = sum(1 for d, _ in done_experiments if d == disaster)
    n_pend = sum(1 for d, _, _ in pending_experiments if d == disaster)
    n_skip = sum(1 for d, _ in skip_experiments if d == disaster)
    print(f"{disaster:<35} {n_done:>5} {n_pend:>8} {n_skip:>5}")

print("─" * 58)
print(f"Total: {total_experiments} experiments | "
      f"Done: {len(done_experiments)} | Pending: {len(pending_experiments)} | Skip: {len(skip_experiments)}")
print(f"Total entries to process: {total_pending_entries:,}")

Disaster                             Done  Pending  Skip
──────────────────────────────────────────────────────────
california_wildfires_2018               0       12     0
canada_wildfires_2016                   0       12     0
cyclone_idai_2019                       0       12     0
hurricane_dorian_2019                   0       12     0
hurricane_florence_2018                 0       12     0
hurricane_harvey_2017                   0       12     0
hurricane_irma_2017                     0       12     0
hurricane_maria_2017                    0       12     0
kaikoura_earthquake_2016                0       12     0
kerala_floods_2018                      0       12     0
──────────────────────────────────────────────────────────
Total: 120 experiments | Done: 0 | Pending: 120 | Skip: 0
Total entries to process: 760,752


In [ ]:
# --- Run ST on all pending experiments ---
all_results = {}  # key: (disaster, train_file) -> result dict
entries_completed = 0
experiments_completed = 0
experiment_start_time = time.time()

# Load results for already-completed experiments
for disaster, train_file in done_experiments:
    results_path = os.path.join(RESULTS_DIR, disaster, f"{results_file_for(train_file)}.txt")
    with open(results_path) as f:
        all_results[(disaster, train_file)] = json.load(f)
    _, n_cls = detect_classes(disaster, train_file, data_root=DATA_ROOT)
    all_results[(disaster, train_file)]["n_classes"] = n_cls

if not pending_experiments:
    print("Nothing to run — all experiments already completed.")
else:
    next_pct_milestone = 10

    for idx, (disaster, train_file, disaster_entries) in enumerate(pending_experiments, 1):
        label_to_id, n_classes = detect_classes(disaster, train_file, data_root=DATA_ROOT)
        ds_train, ds_dev, ds_test, ds_unlabeled = load_disaster_datasets(
            disaster, train_file, tokenizer, label_to_id, data_root=DATA_ROOT
        )

        print(f"[{idx}/{len(pending_experiments)}] {disaster} / {train_file} "
              f"({n_classes} cls, {disaster_entries:,} entries) ...", end=" ")

        disaster_start = time.time()
        with contextlib.redirect_stdout(io.StringIO()):
            train_model(
                ds_train, ds_dev, ds_test, ds_unlabeled,
                PT_TEACHER_CHECKPOINT, cfg,
                model_dir=disaster,
                sup_batch_size=SUP_BATCH_SIZE,
                unsup_batch_size=UNSUP_BATCH_SIZE,
                unsup_size=UNSUP_SIZE,
                sample_size=SAMPLE_SIZE,
                sample_scheme=SAMPLE_SCHEME,
                T=T,
                alpha=ALPHA,
                sup_epochs=SUP_EPOCHS,
                unsup_epochs=UNSUP_EPOCHS,
                N_base=N_BASE,
                dense_dropout=DENSE_DROPOUT,
                attention_probs_dropout_prob=ATTENTION_PROBS_DROPOUT_PROB,
                hidden_dropout_prob=HIDDEN_DROPOUT_PROB,
                results_file=results_file_for(train_file),
                results_dir=RESULTS_DIR,
                data_dir=DATA_ROOT,
                temp_scaling=False,
                ls=0.0,
                n_classes=n_classes,
            )
        disaster_elapsed = time.time() - disaster_start

        # Collect results
        results_path = os.path.join(RESULTS_DIR, disaster, f"{results_file_for(train_file)}.txt")
        if os.path.exists(results_path):
            with open(results_path) as f:
                result = json.load(f)
            all_results[(disaster, train_file)] = result
            all_results[(disaster, train_file)]["n_classes"] = n_classes
            f1 = result.get("Best ST model", {}).get("F1 before temp scaling", "N/A")
            print(f"F1={f1} ({disaster_elapsed/60:.1f} min)")
        else:
            print(f"no results file ({disaster_elapsed/60:.1f} min)")

        # Update progress
        entries_completed += disaster_entries
        experiments_completed += 1
        pct_done = entries_completed / total_pending_entries * 100
        elapsed_total = time.time() - experiment_start_time
        elapsed_hours = elapsed_total / 3600
        rate = entries_completed / elapsed_total
        entries_remaining = total_pending_entries - entries_completed
        est_remaining_hours = (entries_remaining / rate / 3600) if rate > 0 else 0

        if idx == 1 or pct_done >= next_pct_milestone:
            print(f"  >>> {pct_done:.0f}% done | {experiments_completed}/{len(pending_experiments)} experiments "
                  f"| Elapsed: {elapsed_hours:.2f} hrs | ETA: {est_remaining_hours:.2f} hrs")
            while next_pct_milestone <= pct_done:
                next_pct_milestone += 10

    total_elapsed = (time.time() - experiment_start_time) / 3600
    print(f"\nAll {len(pending_experiments)} experiments complete. Total time: {total_elapsed:.2f} hrs")

  Train: 50 | Dev: 752 | Test: 1461 | Unlabeled: 5113
[1/120] california_wildfires_2018 / 5_set1 (10 cls, 7,376 entries) ... F1=0.35303146160532656 (8.3 min)
  >>> 1% done | 1/120 experiments | Elapsed: 0.14 hrs | ETA: 14.18 hrs
  Train: 50 | Dev: 752 | Test: 1461 | Unlabeled: 5113
[2/120] california_wildfires_2018 / 5_set2 (10 cls, 7,376 entries) ... F1=0.43301708756254503 (8.7 min)
  Train: 50 | Dev: 752 | Test: 1461 | Unlabeled: 5113
[3/120] california_wildfires_2018 / 5_set3 (10 cls, 7,376 entries) ... F1=0.4440318620342973 (10.2 min)
  Train: 100 | Dev: 752 | Test: 1461 | Unlabeled: 5063
[4/120] california_wildfires_2018 / 10_set1 (10 cls, 7,376 entries) ... F1=0.5541229256437166 (9.7 min)
  Train: 100 | Dev: 752 | Test: 1461 | Unlabeled: 5063
[5/120] california_wildfires_2018 / 10_set2 (10 cls, 7,376 entries) ... F1=0.569319398309996 (9.9 min)
  Train: 100 | Dev: 752 | Test: 1461 | Unlabeled: 5063
[6/120] california_wildfires_2018 / 10_set3 (10 cls, 7,376 entries) ... F1=0.508731

## Results Summary

Aggregate results across all disasters and splits. The table shows macro-F1 for each combination, plus averages by label size (over 3 sets) and by disaster (over all 12 splits).

In [ ]:
# Build detailed results table
rows = []
for (disaster, train_file), result in all_results.items():
    best = result.get("Best ST model", {})
    # Parse size and set from train_file (e.g., "5_set1" -> size=5, set=1)
    parts = train_file.split("_set")
    size = int(parts[0])
    set_num = int(parts[1])
    rows.append({
        "Disaster": disaster,
        "Size": size,
        "Set": set_num,
        "Split": train_file,
        "Classes": result.get("n_classes", ""),
        "F1": pd.to_numeric(best.get("F1 before temp scaling"), errors="coerce"),
        "ECE": pd.to_numeric(best.get("ECE before temp scaling"), errors="coerce"),
    })

if rows:
    df = pd.DataFrame(rows)

    # --- Pivot table: F1 by disaster × size (averaged over 3 sets) ---
    print("=== Mean F1 by Disaster × Label Size (averaged over 3 sets) ===\n")
    pivot = df.pivot_table(values="F1", index="Disaster", columns="Size", aggfunc="mean")
    pivot["Mean"] = pivot.mean(axis=1)
    pivot.loc["Mean"] = pivot.mean(axis=0)
    print(pivot.round(4).to_string())

    # --- Pivot table: ECE by disaster × size ---
    print("\n\n=== Mean ECE by Disaster × Label Size (averaged over 3 sets) ===\n")
    pivot_ece = df.pivot_table(values="ECE", index="Disaster", columns="Size", aggfunc="mean")
    pivot_ece["Mean"] = pivot_ece.mean(axis=1)
    pivot_ece.loc["Mean"] = pivot_ece.mean(axis=0)
    print(pivot_ece.round(4).to_string())

    # --- Detailed per-split table ---
    print(f"\n\n=== Detailed Results ({len(df)} experiments) ===\n")
    df_sorted = df.sort_values(["Disaster", "Size", "Set"])
    print(df_sorted[["Disaster", "Split", "Classes", "F1", "ECE"]].to_string(index=False))
else:
    print("No results collected. Run the experiment cells above first.")